In [ ]:
from collections import Counter
from importlib import reload
import pandas as pd
import numpy as np
import pickle
import math
import csv
import sys
import os
sys.path.append("/u/<username>/local/pyenvs/pymeasurements")
import plot_utils as pu
reload(pu)
%matplotlib inline

## Load US-based RTTs

In [ ]:
ext_rtt_path    = "data/campus_trace_ext_rtt.csv"
flow_map_path   = "data/campus_trace_flow_map.csv"
geoloc_path     = "data/campus_trace_geolocation_map.csv"
ext_ip_map_path = "data/campus_trace_external_ip_map.csv"
int_ip_map_path = "data/campus_trace_internal_ip_map.csv"
conn_bytes_path = "data/conn_bytes_dict.pkl"

In [ ]:
ext_ip_map = {}
with open(ext_ip_map_path) as fp:
    for line in [l.strip() for l in fp.readlines()][1:]:
        tokens = line.split(",")
        ext_ip_map[tokens[0]] = tokens[1]
print(f"No. of external IPs: {len(ext_ip_map)}")

In [ ]:
int_ip_map = {}
with open(int_ip_map_path) as fp:
    for line in [l.strip() for l in fp.readlines()][1:]:
        tokens = line.split(",")
        int_ip_map[tokens[0]] = tokens[1]
print(f"No. of internal IPs: {len(int_ip_map)}")

In [ ]:
conn_bytes = None
with open("data/conn_bytes_dict.pkl", "rb") as fp:
    conn_bytes = pickle.load(fp)
    
def get_conn_bytes(row):
    conn_id = (row['Source_IP'], row['Source_Port'], row['Destination_IP'], row['Destination_Port'])
    return conn_bytes[conn_id]

In [ ]:
# Geolocation map
df_geo = pd.read_csv(geoloc_path, dtype={"Latitude": str, "Longitude": str})
print("df_geo row count:", df_geo.shape[0])
print(df_geo.head(n=1))
# External IPs geolocated to the US
usa_ext_ips = df_geo[df_geo['Country'] == 'United States']['External_IP_ID'].tolist()
print("\nUSA external IPs count:", len(usa_ext_ips))
df_geo = None

def external_host_type(row):
    if row['Source_Port'] > 1023 and row['Destination_Port'] < 1024:
        return 'C' # Client
    if row['Source_Port'] < 1024 and row['Destination_Port'] > 1023:
        return 'S' # Server
    return 'N' # Neither

def get_trace_source_ip(row):
    if row['Source_IP'] in int_ip_map:
        return int_ip_map[row['Source_IP']]

def get_trace_destination_ip(row):
    if row['Destination_IP'] in ext_ip_map:
        return ext_ip_map[row['Destination_IP']]

# Flows
df_flows = pd.read_csv(flow_map_path)
print("\ndf_flows row count:", df_flows.shape[0])
print(df_flows.head(n=1))
df_flows_usa = df_flows[df_flows['Destination_IP'].isin(usa_ext_ips)]
df_flows_usa['Source_IP'] = df_flows_usa.apply(get_trace_source_ip, axis=1)
df_flows_usa['Destination_IP'] = df_flows_usa.apply(get_trace_destination_ip, axis=1)
df_flows_usa['External_Host_Type'] = df_flows_usa.apply(external_host_type, axis=1)
df_flows_usa['Connection_Bytes'] = df_flows_usa.apply(get_conn_bytes, axis=1)
df_flows_usa['Destination_Prefix'] = df_flows_usa['Destination_IP'].apply(lambda x: ".".join(x.split(".")[:-1]+["0"]))
print(f"\ndf_flows_usa row count: {df_flows_usa.shape[0]} ({round(df_flows_usa.shape[0]*100.0/df_flows.shape[0], 1)}%)")
print(df_flows_usa.head(n=1))
df_flows = None

In [ ]:
df_flows_usa_puclients = df_flows_usa[df_flows_usa['External_Host_Type'] == 'C']
df_flows_usa_puservers = df_flows_usa[df_flows_usa['External_Host_Type'] == 'S']
df_flows_usa_puneither = df_flows_usa[df_flows_usa['External_Host_Type'] == 'N']

In [ ]:
def combine_host_types(row):
    return list(set(row['External_Host_Type']))

df_prefixes_usa = df_flows_usa.groupby('Destination_Prefix').agg({
    'Connection_Bytes': 'sum',
    'Flow_ID': list,
    'Source_IP': list,
    'Destination_IP': list,
    'Source_Port': list,
    'Destination_Port': list,
    'External_Host_Type': list
}).reset_index()
df_prefixes_usa['External_Host_Types'] = df_prefixes_usa.apply(combine_host_types, axis=1)
print(f"Shape of df_prefixes_usa: {df_prefixes_usa.shape[0]}")
print(df_prefixes_usa.head(n=1))

In [ ]:
df_prefixes_usa_puclients = df_prefixes_usa[df_prefixes_usa['External_Host_Types'].apply(lambda x: len(x) == 1 and x[0] == 'C')]
df_prefixes_usa_puservers = df_prefixes_usa[df_prefixes_usa['External_Host_Types'].apply(lambda x: len(x) == 1 and x[0] == 'S')]
df_prefixes_usa_pumixed   = df_prefixes_usa[df_prefixes_usa['External_Host_Types'].apply(lambda x: len(x) == 2 and 'C' in x and 'S' in x)]
df_prefixes_usa_puother   = df_prefixes_usa[df_prefixes_usa['External_Host_Types'].apply(lambda x: 'N' in x)]

In [ ]:
bytes_prefix_puclients = dict(zip(df_prefixes_usa_puclients['Destination_Prefix'], df_prefixes_usa_puclients['Connection_Bytes']))
bytes_prefix_puservers = dict(zip(df_prefixes_usa_puservers['Destination_Prefix'], df_prefixes_usa_puservers['Connection_Bytes']))
bytes_prefix_pumixed   = dict(zip(df_prefixes_usa_pumixed['Destination_Prefix'], df_prefixes_usa_pumixed['Connection_Bytes']))
bytes_prefix_puother   = dict(zip(df_prefixes_usa_puother['Destination_Prefix'], df_prefixes_usa_puother['Connection_Bytes']))

In [ ]:
# RTTs
df_rtts = pd.read_csv(ext_rtt_path)
print("df_rtts row count:", df_rtts.shape[0])
print(df_rtts.head(n=1))

# RTTs from USA-based flows
df_rtts_usa = df_rtts[df_rtts['Flow_ID'].isin(df_flows_usa['Flow_ID'].unique().tolist())]
print(f"\ndf_rtts_usa row count: {df_rtts_usa.shape[0]} ({round(df_rtts_usa.shape[0]*100.0/df_rtts.shape[0], 2)}%)")
print(df_rtts_usa.head(n=1))
df_rtts = None

In [ ]:
df_rtts_flows_usa_puclients = df_rtts_usa[df_rtts_usa['Flow_ID'].isin(df_flows_usa_puclients['Flow_ID'].unique().tolist())]
df_rtts_flows_usa_puservers = df_rtts_usa[df_rtts_usa['Flow_ID'].isin(df_flows_usa_puservers['Flow_ID'].unique().tolist())]
df_rtts_flows_usa_puneither = df_rtts_usa[df_rtts_usa['Flow_ID'].isin(df_flows_usa_puneither['Flow_ID'].unique().tolist())]

len_flow = len(df_rtts_usa['Flow_ID'].unique().tolist())
len_flow_puclients = len(df_rtts_flows_usa_puclients['Flow_ID'].unique().tolist())
len_flow_puservers = len(df_rtts_flows_usa_puservers['Flow_ID'].unique().tolist())
len_flow_puneither = len(df_rtts_flows_usa_puneither['Flow_ID'].unique().tolist())

print(f"No. of flows in per-flow PU clients dataframe: {len_flow_puclients} ({round(len_flow_puclients*100/len_flow, 2)}%)")
print(f"No. of flows in per-flow PU servers dataframe: {len_flow_puservers} ({round(len_flow_puservers*100/len_flow, 2)}%)")
print(f"No. of flows in per-flow PU neither dataframe: {len_flow_puneither} ({round(len_flow_puneither*100/len_flow, 2)}%)")

In [ ]:
# Load into per-flow dicts
rtts_flow_puclients = df_rtts_flows_usa_puclients.groupby('Flow_ID')[[
    'ACK_Timestamp', 'RTT_ms']].apply(lambda g: g.values.tolist()).to_dict()
rtts_flow_puservers = df_rtts_flows_usa_puservers.groupby('Flow_ID')[[
    'ACK_Timestamp', 'RTT_ms']].apply(lambda g: g.values.tolist()).to_dict()
rtts_flow_puneither = df_rtts_flows_usa_puneither.groupby('Flow_ID')[[
    'ACK_Timestamp', 'RTT_ms']].apply(lambda g: g.values.tolist()).to_dict()

print("No. of flow IDs in per-flow PU clients dict:", len(list(rtts_flow_puclients.keys())))
print("No. of flow IDs in per-flow PU servers dict:", len(list(rtts_flow_puservers.keys())))
print("No. of flow IDs in per-flow PU neither dict:", len(list(rtts_flow_puneither.keys())))

for flowid in rtts_flow_puclients:
    rtts_flow_puclients[flowid] = sorted(rtts_flow_puclients[flowid], key = lambda x: x[0])
for flowid in rtts_flow_puservers:
    rtts_flow_puservers[flowid] = sorted(rtts_flow_puservers[flowid], key = lambda x: x[0])
for flowid in rtts_flow_puneither:
    rtts_flow_puneither[flowid] = sorted(rtts_flow_puneither[flowid], key = lambda x: x[0])

In [ ]:
# Prefix-aggregated RTTs
df_rtts_prefixes_usa = pd.merge(df_rtts_usa, df_flows_usa, on='Flow_ID', how='left')[[
    'Destination_Prefix', 'Flow_ID', 'External_Host_Type', 'ACK_Timestamp', 'RTT_ms']]
print(f"df_rtts_prefixes_usa row count: {df_rtts_prefixes_usa.shape[0]} ({round(df_rtts_prefixes_usa.shape[0]*100/df_rtts_usa.shape[0])}%)")
print(df_rtts_prefixes_usa.head(n=1))
df_rtts_usa = None

In [ ]:
df_rtts_prefixes_usa_puclients = df_rtts_prefixes_usa[df_rtts_prefixes_usa['Destination_Prefix'].isin(df_prefixes_usa_puclients['Destination_Prefix'])]
df_rtts_prefixes_usa_puservers = df_rtts_prefixes_usa[df_rtts_prefixes_usa['Destination_Prefix'].isin(df_prefixes_usa_puservers['Destination_Prefix'])]
df_rtts_prefixes_usa_pumixed   = df_rtts_prefixes_usa[df_rtts_prefixes_usa['Destination_Prefix'].isin(df_prefixes_usa_pumixed['Destination_Prefix'])]
df_rtts_prefixes_usa_puother   = df_rtts_prefixes_usa[df_rtts_prefixes_usa['Destination_Prefix'].isin(df_prefixes_usa_puother['Destination_Prefix'])]

len_prefix = len(df_rtts_prefixes_usa['Destination_Prefix'].unique().tolist())
len_prefix_puclients = len(df_rtts_prefixes_usa_puclients['Destination_Prefix'].unique().tolist())
len_prefix_puservers = len(df_rtts_prefixes_usa_puservers['Destination_Prefix'].unique().tolist())
len_prefix_pumixed = len(df_rtts_prefixes_usa_pumixed['Destination_Prefix'].unique().tolist())
len_prefix_puother = len(df_rtts_prefixes_usa_puother['Destination_Prefix'].unique().tolist())

print(f"No. of prefixes in per-prefix PU clients dataframe: {len_prefix_puclients} ({round(len_prefix_puclients*100/len_prefix, 2)}%)")
print(f"No. of prefixes in per-prefix PU servers dataframe: {len_prefix_puservers} ({round(len_prefix_puservers*100/len_prefix, 2)}%)")
print(f"No. of prefixes in per-prefix PU mixed dataframe: {len_prefix_pumixed} ({round(len_prefix_pumixed*100/len_prefix, 2)}%)")
print(f"No. of prefixes in per-prefix PU other dataframe: {len_prefix_puother} ({round(len_prefix_puother*100/len_prefix, 2)}%)")

In [ ]:
# Load into per-prefix dicts
rtts_prefix_puclients = df_rtts_prefixes_usa_puclients.groupby('Destination_Prefix')[[
    'Flow_ID', 'ACK_Timestamp', 'RTT_ms']].apply(lambda g: g.values.tolist()).to_dict()
rtts_prefix_puservers = df_rtts_prefixes_usa_puservers.groupby('Destination_Prefix')[[
    'Flow_ID', 'ACK_Timestamp', 'RTT_ms']].apply(lambda g: g.values.tolist()).to_dict()
rtts_prefix_pumixed = df_rtts_prefixes_usa_pumixed.groupby('Destination_Prefix')[[
    'Flow_ID', 'ACK_Timestamp', 'RTT_ms']].apply(lambda g: g.values.tolist()).to_dict()
rtts_prefix_puother = df_rtts_prefixes_usa_puother.groupby('Destination_Prefix')[[
    'Flow_ID', 'ACK_Timestamp', 'RTT_ms']].apply(lambda g: g.values.tolist()).to_dict()

print("No. of prefixes in per-prefix PU clients dict:", len(list(rtts_prefix_puclients.keys())))
print("No. of prefixes in per-prefix PU servers dict:", len(list(rtts_prefix_puservers.keys())))
print("No. of prefixes in per-prefix PU mixed dict:", len(list(rtts_prefix_pumixed.keys())))
print("No. of prefixes in per-prefix PU other dict:", len(list(rtts_prefix_puother.keys())))

for prefix in rtts_prefix_puclients:
    rtts_prefix_puclients[prefix] = sorted(rtts_prefix_puclients[prefix], key = lambda x: x[1])
for prefix in rtts_prefix_puservers:
    rtts_prefix_puservers[prefix] = sorted(rtts_prefix_puservers[prefix], key = lambda x: x[1])
for prefix in rtts_prefix_pumixed:
    rtts_prefix_pumixed[prefix] = sorted(rtts_prefix_pumixed[prefix], key = lambda x: x[1])
for prefix in rtts_prefix_puother:
    rtts_prefix_puother[prefix] = sorted(rtts_prefix_puother[prefix], key = lambda x: x[1])

### Store US-based bytes and RTTs

In [ ]:
bytes_prefix_puclients_path = "data/bytes_prefix_puclients.pkl"
bytes_prefix_puservers_path = "data/bytes_prefix_puservers.pkl"
bytes_prefix_pumixed_path = "data/bytes_prefix_pumixed.pkl"
bytes_prefix_puother_path = "data/bytes_prefix_puother.pkl"

In [ ]:
rtts_prefix_puclients_path = "data/rtts_prefix_puclients.pkl"
rtts_prefix_puservers_path = "data/rtts_prefix_puservers.pkl"
rtts_prefix_pumixed_path = "data/rtts_prefix_pumixed.pkl"
rtts_prefix_puother_path = "data/rtts_prefix_puother.pkl"

In [ ]:
def dump_data_dict(data_dict, data_path):
    with open(data_path, "wb") as fp:
        pickle.dump(data_dict, fp)

In [ ]:
dump_data_dict(bytes_prefix_puclients, bytes_prefix_puclients_path)
dump_data_dict(bytes_prefix_puservers, bytes_prefix_puservers_path)
dump_data_dict(bytes_prefix_pumixed, bytes_prefix_pumixed_path)
dump_data_dict(bytes_prefix_puother, bytes_prefix_puother_path)

In [ ]:
dump_data_dict(rtts_prefix_puclients, rtts_prefix_puclients_path)
dump_data_dict(rtts_prefix_puservers, rtts_prefix_puservers_path)
dump_data_dict(rtts_prefix_pumixed, rtts_prefix_pumixed_path)
dump_data_dict(rtts_prefix_puother, rtts_prefix_puother_path)

### Load US-based RTTs

In [ ]:
def load_data_dict(data_path):
    with open(data_path, "rb") as fp:
        return pickle.load(fp)

In [ ]:
bytes_prefix_puclients = load_data_dict(bytes_prefix_puclients_path)
bytes_prefix_puservers = load_data_dict(bytes_prefix_puservers_path)
bytes_prefix_pumixed   = load_data_dict(bytes_prefix_pumixed_path)
bytes_prefix_puother   = load_data_dict(bytes_prefix_puother_path)

In [ ]:
rtts_prefix_puclients = load_data_dict(rtts_prefix_puclients_path)
rtts_prefix_puservers = load_data_dict(rtts_prefix_puservers_path)
rtts_prefix_pumixed   = load_data_dict(rtts_prefix_pumixed_path)
rtts_prefix_puother   = load_data_dict(rtts_prefix_puother_path)

In [ ]:
pu.barplot(["Client\nin PU", "Server\nin PU", "Both", "None"],
           [len(rtts_prefix_puclients)/1000, len(rtts_prefix_puservers)/1000, len(rtts_prefix_pumixed)/1000, len(rtts_prefix_puother)/1000],
           title="Prefix count per type", xlabel="Prefix type", ylabel="Prefix count (K)")

## Surged prefixes

### RTT windowing, min-taking, filtering functions

In [ ]:
def divide_into_windows_with_reltime(flow_list, time_list_abs, rtt_list, window_size):
    windows_flow = []
    windows_time = []
    windows_rtt = []
    current_window_flow = []
    current_window_time = []
    current_window_rtt = []

    utc_time_min = math.floor(time_list_abs[0])
    time_list = [(t - utc_time_min) for t in time_list_abs]
    
    for flow, time, rtt in zip(flow_list, time_list, rtt_list):
        if not current_window_time:
            current_window_flow.append(flow)
            current_window_time.append(time)
            current_window_rtt.append(rtt)
        else:
            window_start = current_window_time[0]
            window_number = math.floor(time / window_size)
            start_window_number = math.floor(window_start / window_size)
            
            if window_number == start_window_number:
                current_window_flow.append(flow)
                current_window_time.append(time)
                current_window_rtt.append(rtt)
            else:
                windows_flow.append(current_window_flow)
                windows_time.append(current_window_time)
                windows_rtt.append(current_window_rtt)
                current_window_flow = [flow]
                current_window_time = [time]
                current_window_rtt = [rtt]
    
    if current_window_time:
        windows_flow.append(current_window_flow)
        windows_time.append(current_window_time)
        windows_rtt.append(current_window_rtt)
    
    return windows_flow, windows_time, windows_rtt

In [ ]:
def extract_minimums_and_window_len(flows, tstamps, rtts):
    tstamp_mins = []
    rtt_mins = []
    window_lens = []
    flow_lens = []
    for f, t, r in zip(flows, tstamps, rtts):
        loc = r.index(min(r))
        tstamp_mins.append(t[loc])
        rtt_mins.append(r[loc])
        window_lens.append(len(r))
        flow_lens.append({fid: f.count(fid) for fid in set(f)})
    return tstamp_mins, rtt_mins, window_lens, flow_lens

In [ ]:
def minrtts(rtts_dict, prefix, win_size_time, win_size_samples):
    fid_series = [i[0] for i in rtts_dict[prefix]]
    ts_series  = [i[1] for i in rtts_dict[prefix]]
    tsr_series = [(i - math.floor(ts_series[0])) for i in ts_series]
    rtt_series = [i[2] for i in rtts_dict[prefix]]
    flows_1, time_1, rtt_1 = divide_into_windows_with_reltime(fid_series, ts_series, rtt_series, win_size_time)
    min_time, min_rtts, win_len, flows_len = extract_minimums_and_window_len(flows_1, time_1, rtt_1)

    x_min_time = []
    y_min_rtts = []
    for t, r, w in zip(min_time, min_rtts, win_len):
        if w >= win_size_samples:
            x_min_time.append(t)
            y_min_rtts.append(r)

    return tsr_series, x_min_time, rtt_series, y_min_rtts

In [ ]:
def filter_minimums(time, rtt):
    tstamps = []
    minrtts = []
    for t1, t2, r1, r2, w1, w2 in zip(time[:-1], time[1:], rtt[:-1], rtt[1:], winlen[:-1], winlen[1:]):
        if w1 >= window_size_samples and w2 >= window_size_samples:
            tstamps.append(t1)
            minrtts.append(r1)
    if winlen[-1] >= window_size_samples:
        tstamps.append(time[-1])
        minrtts.append(rtt[-1])
    return tstamps, minrtts

## Minimum RTTs per prefix

In [ ]:
def minimums_per_prefix(rtts_dict):
    rtt_mins = []
    for prefix in rtts_dict:
        rtt_mins.append(min([i[2] for i in rtts_dict[prefix]]))
    return rtt_mins

In [ ]:
def minimums_per_prefix_filtered(rtts_dict, window_size_time, window_size_samples):
    rtt_mins = []
    for prefix in rtts_dict:
        ts_series = [i[1] for i in rtts_dict[prefix]]
        tsr_series = [(i - math.floor(ts_series[0])) for i in ts_series]
        rtt_series = [i[2] for i in rtts_dict[prefix]]
        time_1, rtt_1 = divide_into_windows_with_reltime(ts_series, rtt_series, window_size_time)
        min_time, min_rtts, win_len = extract_minimums_and_window_len(time_1, rtt_1)
        if len([w for w in win_len if w >= window_size_samples]) >= 3:
            rtt_mins.append(min(rtt_series))
    return rtt_mins

In [ ]:
mins_puclients = minimums_per_prefix(rtts_prefix_puclients)
mins_puservers = minimums_per_prefix(rtts_prefix_puservers)
mins_pumixed = minimums_per_prefix(rtts_prefix_pumixed)
mins_puother = minimums_per_prefix(rtts_prefix_puother)

In [ ]:
window_size_time = 0.5
window_size_samples = 5

mins_filtered_puclients = minimums_per_prefix_filtered(rtts_prefix_puclients, window_size_time, window_size_samples)
mins_filtered_puservers = minimums_per_prefix_filtered(rtts_prefix_puservers, window_size_time, window_size_samples)
mins_filtered_pumixed = minimums_per_prefix_filtered(rtts_prefix_pumixed, window_size_time, window_size_samples)
mins_filtered_puother = minimums_per_prefix_filtered(rtts_prefix_puother, window_size_time, window_size_samples)

In [ ]:
def report_gt_value(min_rtts, mtype, val=100):
    count = len([i for i in min_rtts if i > val])
    print(f"No. of prefixes of PU {mtype} type: {count} out of {len(min_rtts)} ({round(count*100.0/len(min_rtts), 2)}%)")

In [ ]:
print("Filter with minRTT greater than equal to 100 ms:")
report_gt_value(mins_puclients, "Clients", 100)
report_gt_value(mins_puservers, "Servers", 100)
report_gt_value(mins_pumixed, "Mixed", 100)
report_gt_value(mins_puother, "Others", 100)

print("\nFilter with minRTT greater than equal to 200 ms:")
report_gt_value(mins_puclients, "Clients", 200)
report_gt_value(mins_puservers, "Servers", 200)
report_gt_value(mins_pumixed, "Mixed", 200)
report_gt_value(mins_puother, "Others", 200)

print("\nFilter with minRTT greater than equal to 315 ms:")
report_gt_value(mins_puclients, "Clients", 315)
report_gt_value(mins_puservers, "Servers", 315)
report_gt_value(mins_pumixed, "Mixed", 315)
report_gt_value(mins_puother, "Others", 315)

print("\nFilter with minRTT greater than equal to 630 ms:")
report_gt_value(mins_puclients, "Clients", 630)
report_gt_value(mins_puservers, "Servers", 630)
report_gt_value(mins_pumixed, "Mixed", 630)
report_gt_value(mins_puother, "Others", 630)

In [ ]:
print("Filter with minRTT greater than equal to 100 ms:")
report_gt_value(mins_filtered_puclients, "Clients", 100)
report_gt_value(mins_filtered_puservers, "Servers", 100)
report_gt_value(mins_filtered_pumixed, "Mixed", 100)
report_gt_value(mins_filtered_puother, "Others", 100)

print("\nFilter with minRTT greater than equal to 200 ms:")
report_gt_value(mins_filtered_puclients, "Clients", 200)
report_gt_value(mins_filtered_puservers, "Servers", 200)
report_gt_value(mins_filtered_pumixed, "Mixed", 200)
report_gt_value(mins_filtered_puother, "Others", 200)

print("\nFilter with minRTT greater than equal to 315 ms:")
report_gt_value(mins_filtered_puclients, "Clients", 315)
report_gt_value(mins_filtered_puservers, "Servers", 315)
report_gt_value(mins_filtered_pumixed, "Mixed", 315)
report_gt_value(mins_filtered_puother, "Others", 315)

print("\nFilter with minRTT greater than equal to 630 ms:")
report_gt_value(mins_filtered_puclients, "Clients", 630)
report_gt_value(mins_filtered_puservers, "Servers", 630)
report_gt_value(mins_filtered_pumixed, "Mixed", 630)
report_gt_value(mins_filtered_puother, "Others", 630)

In [ ]:
pu.plot_cdfs([mins_puclients, mins_puservers, mins_pumixed, mins_puother],
             curvelabels=["Client in PU", "Server in PU", "Both", "None"],
             xlabel="RTT (ms)",
             title="MinRTT per Prefix", xscale="log")

In [ ]:
pu.plot_cdfs([mins_filtered_puclients, mins_filtered_puservers, mins_filtered_pumixed, mins_filtered_puother],
             curvelabels=[f"PU Clients ({len(mins_filtered_puclients)//1000}K)", f"PU Servers ({len(mins_filtered_puservers)//1000}K)",
                          f"Mixed ({len(mins_filtered_pumixed)//1000}K)", f"Others ({len(mins_filtered_puother)//1000}K)"],
             xlabel="RTT (ms)", loc="upper left",
             title="MinRTT per Prefix (Filtered)", xscale="log")

## Two window and three window based detection

In [ ]:
window_size_time = 0.5
window_size_samples = 5
surge_thresh = 100
stability_thresh = 20

In [ ]:
def find_prefixes_with_surge(rtts_dict, win_size_time, win_size_samples, surge_thresh):

    surged_prefixes = []
    surge_points = {}

    for prefix in rtts_dict:
        fid_series = [i[0] for i in rtts_dict[prefix]]
        ts_series  = [i[1] for i in rtts_dict[prefix]]
        tsr_series = [(i - math.floor(ts_series[0])) for i in ts_series]
        rtt_series = [i[2] for i in rtts_dict[prefix]]
        flows_1, time_1, rtt_1 = divide_into_windows_with_reltime(fid_series, ts_series, rtt_series, win_size_time)
        min_time, min_rtts, win_len, flows_len = extract_minimums_and_window_len(flows_1, time_1, rtt_1)
        
        for idx, (t1, t2, r1, r2, w1, w2) in enumerate(zip(min_time[:-1], min_time[1:], min_rtts[:-1], min_rtts[1:], win_len[:-1], win_len[1:])):
            if t2 - t1 <= 2 * win_size_time and w1 >= win_size_samples and w2 >= win_size_samples:
                if r2 - r1 >= surge_thresh:
                    surged_prefixes.append(prefix)
                    surge_points[prefix] = [(idx, t1, r1), (idx+1, t2, r2)]
                    break

    return surged_prefixes, surge_points

In [ ]:
def find_prefixes_with_surge_and_stability(rtts_dict, win_size_time, win_size_samples, surge_thresh, stability_thresh):

    attacked_prefixes = []
    attack_points = {}

    for prefix in rtts_dict:
        fid_series = [i[0] for i in rtts_dict[prefix]]
        ts_series  = [i[1] for i in rtts_dict[prefix]]
        tsr_series = [(i - math.floor(ts_series[0])) for i in ts_series]
        rtt_series = [i[2] for i in rtts_dict[prefix]]
        flows_1, time_1, rtt_1 = divide_into_windows_with_reltime(fid_series, ts_series, rtt_series, win_size_time)
        min_time, min_rtts, win_len, flows_len = extract_minimums_and_window_len(flows_1, time_1, rtt_1)
        
        for idx, (t1, t2, t3, r1, r2, r3, w1, w2, w3) in enumerate(zip(
                min_time[:-2], min_time[1:-1], min_time[2:], min_rtts[:-2], min_rtts[1:-1], min_rtts[2:], win_len[:-2], win_len[1:-1], win_len[2:])):
            if t2 - t1 <= 2 * win_size_time and t3 - t2 <= 2 * win_size_time and t3 - t1 <= 3 * win_size_time \
                    and w1 >= win_size_samples and w2 >= win_size_samples and w3 >= win_size_samples:
                if r2 - r1 >= surge_thresh and abs(r3 - r2) <= stability_thresh:
                    attacked_prefixes.append(prefix)
                    # print(f"{prefix}: ({t1}, {r1}, {w1}), ({t2}, {r2}, {w2}), ({t3}, {r3}, {w3})")
                    attack_points[prefix] = [(idx, t1, r1), (idx+1, t2, r2), (idx+2, t3, r3)]
                    break

    return attacked_prefixes, attack_points

### Client in PU

In [ ]:
def add_prefix_bytes(prefixes, bytes_dict):
    sum_bytes = 0
    total_bytes = 0
    for prefix in bytes_dict:
        total_bytes += bytes_dict[prefix]
        if prefix in prefixes:
            sum_bytes += bytes_dict[prefix]
    return sum_bytes, total_bytes

In [ ]:
surged_prefixes_puclients, surge_points_puclients = find_prefixes_with_surge(
    rtts_prefix_puclients, window_size_time, window_size_samples, surge_thresh)

In [ ]:
print(f"No. of surged prefixes: {len(surged_prefixes_puclients)}")
surged_bytes_puclients, total_bytes_puclients = add_prefix_bytes(surged_prefixes_puclients, bytes_prefix_puclients)
print(f"Bytes in surged prefixes: {round(surged_bytes_puclients/1000000000, 2)} GB ({round(surged_bytes_puclients*100.0/total_bytes_puclients, 2)}%)\n")
print("Surged prefixes:")
for prefix in sorted(surged_prefixes_puclients):
    print(f"\tPrefix: {prefix}/24, Surge points: {surge_points_puclients[prefix]}")

In [ ]:
attacked_prefixes_puclients, attack_points_puclients = find_prefixes_with_surge_and_stability(
    rtts_prefix_puclients, window_size_time, window_size_samples, surge_thresh, stability_thresh)

In [ ]:
print(f"No. of attacked prefixes: {len(attacked_prefixes_puclients)}")
attacked_bytes_puclients, total_bytes_puclients = add_prefix_bytes(attacked_prefixes_puclients, bytes_prefix_puclients)
print(f"Bytes in attacked prefixes: {round(attacked_bytes_puclients/1000000000, 2)} GB ({round(attacked_bytes_puclients*100.0/total_bytes_puclients, 3)}%)\n")
print("Attacked prefixes:")
for prefix in sorted(attacked_prefixes_puclients):
    print(f"\tPrefix: {prefix}/24, Attack points: {attack_points_puclients[prefix]}")

In [ ]:
for prefix in sorted(attacked_prefixes_puclients):
    tsr_series, min_time, rtt_series, min_rtts = minrtts(rtts_prefix_puclients, prefix, window_size_time, window_size_samples)

    pu.scatterplot(tsr_series, rtt_series,
                    xlabel="Time (s)", ylabel="RTT (ms)",
                    title=f"RTT vs. Time ({prefix}/24)", plot_path=f"plots/puclients_fp__rtt_{prefix}.png")
    
    pu.scatterplot(min_time, min_rtts,
                    xlabel="Time (s)", ylabel="MinRTT (ms)",
                    title=f"MinRTT vs. Time ({prefix}/24)", plot_path=f"plots/puclients_fp__minrtt_{prefix}.png")

    pu.scatterplots([tsr_series, [i[0] for i in attack_points_puclients[prefix]]], [rtt_series, [i[1] for i in attack_points_puclients[prefix]]],
                    xlim = (attack_points_puclients[prefix][0][0] - 5.0, attack_points_puclients[prefix][-1][0] + 5.0), ylim=(-1, max(min_rtts)),
                    curvelabels = ["All samples", "Attack points"],
                    loc = "lower right",
                    xlabel="Time (s)", ylabel="RTT (ms)",
                    title=f"RTT vs. Time ({prefix}/24)", plot_path=f"plots/puclients_fp__attackpts_{prefix}.png")
    
    print("\n\n")
    # break

### Server in PU

In [ ]:
surged_prefixes_puservers, surge_points_puservers = find_prefixes_with_surge(
    rtts_prefix_puservers, window_size_time, window_size_samples, surge_thresh)

In [ ]:
print(f"No. of surged prefixes: {len(surged_prefixes_puservers)}")
surged_bytes_puservers, total_bytes_puservers = add_prefix_bytes(surged_prefixes_puservers, bytes_prefix_puservers)
print(f"Bytes in surged prefixes: {surged_bytes_puservers/1000000000} GB ({round(surged_bytes_puservers*100.0/total_bytes_puservers, 2)}%)\n")
print("Surged prefixes:")
for prefix in sorted(surged_prefixes_puservers):
    print(f"\tSurge points: {surge_points_puservers[prefix]}")

In [ ]:
attacked_prefixes_puservers, attack_points_puservers = find_prefixes_with_surge_and_stability(
    rtts_prefix_puservers, window_size_time, window_size_samples, surge_thresh, stability_thresh)

In [ ]:
print(f"No. of attacked prefixes: {len(attacked_prefixes_puservers)}")
attacked_bytes_puservers, total_bytes_puservers = add_prefix_bytes(attacked_prefixes_puservers, bytes_prefix_puservers)
print(f"Bytes in attacked prefixes: {attacked_bytes_puservers/1000000000} GB ({round(attacked_bytes_puservers*100.0/total_bytes_puservers, 2)}%)\n")
print("Attacked prefixes:")
for prefix in sorted(attacked_prefixes_puservers):
    print(f"\tAttack points: {attack_points_puservers[prefix]}")

In [ ]:
for prefix in sorted(attacked_prefixes_puservers)[:5]:
    tsr_series, min_time, rtt_series, min_rtts = minrtts(rtts_prefix_puservers, prefix, window_size_time, window_size_samples)

    pu.scatterplot(tsr_series, rtt_series,
                    xlabel="Time (s)", ylabel="RTT (ms)",
                    title=f"RTT vs. Time ({prefix}/24)", plot_path=f"plots/now_puservers_fp__rtt_{prefix}.png")
    
    pu.scatterplot(min_time, min_rtts,
                    xlabel="Time (s)", ylabel="MinRTT (ms)",
                    title=f"MinRTT vs. Time ({prefix}/24)", plot_path=f"plots/now_puservers_fp__minrtt_{prefix}.png")

    pu.scatterplots([tsr_series, [i[0] for i in attack_points_puservers[prefix]]], [rtt_series, [i[1] for i in attack_points_puservers[prefix]]],
                    xlim = (attack_points_puservers[prefix][0][0] - 5.0, attack_points_puservers[prefix][-1][0] + 5.0), ylim=(-1, max(min_rtts)),
                    curvelabels = ["All samples", "Attack points"],
                    loc = "upper right",
                    xlabel="Time (s)", ylabel="RTT (ms)",
                    title=f"RTT vs. Time ({prefix}/24)", plot_path=f"plots/now_puservers_fp__attackpts_{prefix}.png")
    
    print("\n\n")
    # break

## Analyze cost of dropping packets upon attack detection

In [ ]:
window_size_time = 0.5
window_size_samples = 5
surge_thresh = 100
stability_thresh = 20

In [ ]:
def compute_active_time(time_series, start_time, end_time, win_size):
    total_active_time = 0
    last_window_end = 0
    
    for timestamp in time_series:
        if timestamp >= start_time and timestamp <= end_time:
            # print(timestamp)
            current_window_end = (timestamp // win_size + 1) * win_size
            # print(current_window_end)
            if current_window_end != last_window_end:
                total_active_time += win_size
                # print(total_active_time)
                last_window_end = current_window_end

    return total_active_time

In [ ]:
def compute_total_active_time_for_all_prefixes(rtts_dict, win_size_time):
    total_active_time = 0
    
    for prefix in rtts_dict:
        fid_series = [i[0] for i in rtts_dict[prefix]]
        ts_series  = [i[1] for i in rtts_dict[prefix]]
        tsr_series = [(i - math.floor(ts_series[0])) for i in ts_series]
        rtt_series = [i[2] for i in rtts_dict[prefix]]
        flows_1, time_1, rtt_1 = divide_into_windows_with_reltime(fid_series, ts_series, rtt_series, win_size_time)
        min_time, min_rtts, win_len, flows_len = extract_minimums_and_window_len(flows_1, time_1, rtt_1)

        # Calculate total active time (exclude windows with no minRTT reports)
        total_active_time += compute_active_time(min_time, min_time[0], min_time[-1], win_size_time)

    return total_active_time

In [ ]:
def compute_cost_for_attacked_prefixes(rtts_dict, attacked_prefixes, win_size_time, win_size_samples, surge_thresh, stability_thresh):

    attack_periods = {}

    for prefix in attacked_prefixes:
        fid_series = [i[0] for i in rtts_dict[prefix]]
        ts_series  = [i[1] for i in rtts_dict[prefix]]
        tsr_series = [(i - math.floor(ts_series[0])) for i in ts_series]
        rtt_series = [i[2] for i in rtts_dict[prefix]]
        flows_1, time_1, rtt_1 = divide_into_windows_with_reltime(fid_series, ts_series, rtt_series, win_size_time)
        min_time, min_rtts, win_len, flows_len = extract_minimums_and_window_len(flows_1, time_1, rtt_1)

        # Calculate total active time (exclude windows with no minRTT reports)
        total_active_time = compute_active_time(min_time, min_time[0], min_time[-1], win_size_time)
        status = "normal"

        i = 0
        while i < len(min_rtts) - 2:
            t1, t2, t3 = min_time[i], min_time[i+1], min_time[i+2]
            r1, r2, r3 = min_rtts[i], min_rtts[i+1], min_rtts[i+2]
            w1, w2, w3 = win_len[i], win_len[i+1], win_len[i+2]
            
            if t2 - t1 <= 2 * win_size_time and t3 - t2 <= 2 * win_size_time and t3 - t1 <= 3 * win_size_time \
                and w1 >= win_size_samples and w2 >= win_size_samples and w3 >= win_size_samples:
                if r2 - r1 >= surge_thresh and abs(r3 - r2) <= stability_thresh:
                    state = "attacked"
                    if prefix not in attack_periods:
                        attack_periods[prefix] = []
                    attack_time = (t3//win_size_time + 1) * win_size_time

                    # Find recovery time
                    for j in range(i+3, len(min_rtts)):
                        if min_rtts[j] < r2 - stability_thresh:
                            recovery_time = (min_time[j]//win_size_time + 1) * win_size_time
                            recovery_duration = recovery_time - attack_time
                            recovery_active_time = compute_active_time(min_time, attack_time, recovery_time, win_size_time)
                            attack_periods[prefix].append((
                                attack_time, recovery_time, recovery_duration,
                                recovery_active_time, recovery_active_time*100/total_active_time))
                            i = j  # Move index to after recovery to look for the next attack
                            state = "recovered"
                            break

                    if state == "attacked":
                        recovery_time = None
                        recovery_duration = None
                        recovery_active_time = compute_active_time(min_time, attack_time, min_time[-1], win_size_time)
                        attack_periods[prefix].append((
                            attack_time, recovery_time, recovery_duration,
                            recovery_active_time, recovery_active_time*100/total_active_time))
                            
            i += 1

    return attack_periods

In [ ]:
attack_periods_puclients = compute_cost_for_attacked_prefixes(
    rtts_prefix_puclients, attacked_prefixes_puclients, window_size_time, window_size_samples, surge_thresh, stability_thresh)
attack_periods_puservers = compute_cost_for_attacked_prefixes(
    rtts_prefix_puservers, attacked_prefixes_puservers, window_size_time, window_size_samples, surge_thresh, stability_thresh)

In [ ]:
def compute_total_cost(attack_periods, prefix):
    cost = 0
    if len(attack_periods[prefix]) > 0:
        for period in attack_periods[prefix]:
            cost += period[3]
    return cost

In [ ]:
def compute_total_cost_pct(attack_periods, prefix):
    cost_pct = 0
    if len(attack_periods[prefix]) > 0:
        for period in attack_periods[prefix]:
            cost_pct += period[4]
    return cost_pct

In [ ]:
cost_puclients = [compute_total_cost(attack_periods_puclients, prefix) for prefix in attack_periods_puclients]
cost_puservers = [compute_total_cost(attack_periods_puservers, prefix) for prefix in attack_periods_puservers]

In [ ]:
cost_pct_puclients = [compute_total_cost_pct(attack_periods_puclients, prefix) for prefix in attack_periods_puclients]
cost_pct_puservers = [compute_total_cost_pct(attack_periods_puservers, prefix) for prefix in attack_periods_puservers]

In [ ]:
total_active_time_puclients = compute_total_active_time_for_all_prefixes(rtts_prefix_puclients, window_size_time)
total_active_time_puservers = compute_total_active_time_for_all_prefixes(rtts_prefix_puservers, window_size_time)
print(f"Client in PU: Total active time = {total_active_time_puclients} sec.")
print(f"Server in PU: Total active time = {total_active_time_puservers} sec.")

In [ ]:
total_cost_puclients = sum(cost_puclients)
total_cost_puservers = sum(cost_puservers)
print(f"Client in PU: Total cost = {total_cost_puclients} ({round(total_cost_puclients*100/total_active_time_puclients, 5)}%)")
print(f"Server in PU: Total cost = {total_cost_puservers} ({round(total_cost_puservers*100/total_active_time_puservers, 5)}%)")

In [ ]:
pu.cdf(cost_puclients, xlabel="Cost (secs.)", title="Cost of false positives: Client in PU")

In [ ]:
pu.cdf(cost_puservers, xlabel="Cost (secs.)", title="Cost of false positives: Server in PU", plot_path="plots/cost_fp_puservers.png")

In [ ]:
pu.cdf(cost_puservers, xlabel="Cost (secs.)", title="Cost of false positives: Server in PU", xlim=(-1,101))

In [ ]:
pu.cdf(cost_pct_puservers, xlabel="Cost (%)", title="Cost of false positives: Server in PU", plot_path="plots/cost_pct_fp_puservers.png")

## Two and three window algorithms with multiple flows per window

In [ ]:
window_size_time = 0.5
window_size_samples_total = 5
window_size_numflows = 2
window_size_samples_perflow = 2
surge_thresh = 100
stability_thresh = 20

In [ ]:
def find_prefixes_with_surge_with_minperflowsamples(
        rtts_dict, win_size_time, win_size_samples_total, win_size_numflows, win_size_samples_perflow, surge_thresh):

    surged_prefixes = []
    surge_points = {}

    for prefix_count, prefix in enumerate(rtts_dict):
        flow_series = [i[0] for i in rtts_dict[prefix]]
        ts_series   = [i[1] for i in rtts_dict[prefix]]
        tsr_series  = [(i - math.floor(ts_series[0])) for i in ts_series]
        rtt_series  = [i[2] for i in rtts_dict[prefix]]
        flow_1, time_1, rtt_1 = divide_into_windows_with_reltime(flow_series, ts_series, rtt_series, win_size_time)
        min_time, min_rtts, win_len, flow_len = extract_minimums_and_window_len(flow_1, time_1, rtt_1)
        
        for idx, (t1, t2, r1, r2, w1, w2, f1, f2) in enumerate(
                zip(min_time[:-1], min_time[1:], min_rtts[:-1], min_rtts[1:], win_len[:-1], win_len[1:], flow_len[:-1], flow_len[1:])):
            if t2 - t1 <= 2 * win_size_time and w1 >= win_size_samples_total and w2 >= win_size_samples_total:
                if len(f1) >= win_size_numflows and len(f2) >= win_size_numflows \
                        and sum(1 for i in f1 if f1[i] >= win_size_samples_perflow) >= win_size_numflows \
                        and sum(1 for i in f2 if f2[i] >= win_size_samples_perflow) >= win_size_numflows:
                    if r2 - r1 >= surge_thresh:
                        surged_prefixes.append(prefix)
                        surge_points[prefix] = [(idx, t1, r1), (idx+1, t2, r2)]
                        break

    return surged_prefixes, surge_points

In [ ]:
def find_prefixes_with_surge_and_stability_with_minperflowsamples(
        rtts_dict, win_size_time, win_size_samples_total, win_size_numflows, win_size_samples_perflow, surge_thresh, stability_thresh):

    attacked_prefixes = []
    attack_points = {}

    for prefix in rtts_dict:
        flow_series = [i[0] for i in rtts_dict[prefix]]
        ts_series   = [i[1] for i in rtts_dict[prefix]]
        tsr_series  = [(i - math.floor(ts_series[0])) for i in ts_series]
        rtt_series  = [i[2] for i in rtts_dict[prefix]]
        flow_1, time_1, rtt_1 = divide_into_windows_with_reltime(flow_series, ts_series, rtt_series, win_size_time)
        min_time, min_rtts, win_len, flow_len = extract_minimums_and_window_len(flow_1, time_1, rtt_1)
        
        for idx, (t1, t2, t3, r1, r2, r3, w1, w2, w3, f1, f2, f3) in enumerate(zip(
                min_time[:-2], min_time[1:-1], min_time[2:], min_rtts[:-2], min_rtts[1:-1], min_rtts[2:],
                win_len[:-2], win_len[1:-1], win_len[2:], flow_len[:-2], flow_len[1:-1], flow_len[2:])):
            if t2 - t1 <= 2 * win_size_time and t3 - t2 <= 2 * win_size_time and t3 - t1 <= 3 * win_size_time \
                    and w1 >= win_size_samples_total and w2 >= win_size_samples_total and w3 >= win_size_samples_total:
                if len(f1) >= win_size_numflows and len(f2) >= win_size_numflows and len(f3) >= win_size_numflows \
                        and sum(1 for i in f1 if f1[i] >= win_size_samples_perflow) >= win_size_numflows \
                        and sum(1 for i in f2 if f2[i] >= win_size_samples_perflow) >= win_size_numflows \
                        and sum(1 for i in f3 if f3[i] >= win_size_samples_perflow) >= win_size_numflows:
                    if r2 - r1 >= surge_thresh and abs(r3 - r2) <= stability_thresh:
                        attacked_prefixes.append(prefix)
                        # print(f"{prefix}: ({t1}, {r1}, {w1}), ({t2}, {r2}, {w2}), ({t3}, {r3}, {w3})")
                        attack_points[prefix] = [(idx, t1, r1), (idx+1, t2, r2), (idx+2, t3, r3)]
                        break

    return attacked_prefixes, attack_points

In [ ]:
surged_prefixes_puservers_minsamplesperflow, surge_points_puservers_minsamplesperflow = find_prefixes_with_surge_with_minperflowsamples(
    rtts_prefix_puservers, window_size_time, window_size_samples_total, window_size_numflows, window_size_samples_perflow, surge_thresh)

In [ ]:
print(f"No. of surged prefixes: {len(surged_prefixes_puservers_minsamplesperflow)}")
surged_bytes_puservers_minsamplesperflow, total_bytes_puservers_minsamplesperflow = add_prefix_bytes(
    surged_prefixes_puservers_minsamplesperflow, bytes_prefix_puservers)
print(f"Bytes in surged prefixes: {surged_bytes_puservers_minsamplesperflow/1000000000} GB ("
        + f"{round(surged_bytes_puservers_minsamplesperflow*100.0/total_bytes_puservers_minsamplesperflow, 2)}%)\n")
print("Surged prefixes:")
for prefix in sorted(surged_prefixes_puservers_minsamplesperflow):
    print(f"\tSurge points: {surge_points_puservers_minsamplesperflow[prefix]}")

In [ ]:
attacked_prefixes_puservers_minsamplesperflow, attack_points_puservers_minsamplesperflow = find_prefixes_with_surge_and_stability_with_minperflowsamples(
    rtts_prefix_puservers, window_size_time, window_size_samples_total, window_size_numflows, window_size_samples_perflow, surge_thresh, stability_thresh)

In [ ]:
print(f"No. of attacked prefixes: {len(attacked_prefixes_puservers_minsamplesperflow)}")
attacked_bytes_puservers_minsamplesperflow, total_bytes_puservers_minsamplesperflow = add_prefix_bytes(
    attacked_prefixes_puservers_minsamplesperflow, bytes_prefix_puservers)
print(f"Bytes in attacked prefixes: {attacked_bytes_puservers_minsamplesperflow/1000000000} GB ("
      + f"{round(attacked_bytes_puservers_minsamplesperflow*100.0/total_bytes_puservers, 2)}%)\n")
print("Attacked prefixes:")
for prefix in sorted(attacked_prefixes_puservers_minsamplesperflow):
    print(f"\tAttack points: {attack_points_puservers_minsamplesperflow[prefix]}")

In [ ]:
pu.barplot(["Min. 1 flow\nper window", "Min. 2 flows\nper window"],
           [len(attack_points_puservers), len(attacked_prefixes_puservers_minsamplesperflow)],
           xlabel="Algorithm", ylabel="#Prefixes", title="False positives")

In [ ]:
def compute_cost_for_attacked_prefixes_minsamplesperflow(
    rtts_dict, attacked_prefixes, win_size_time, win_size_samples_total, win_size_numflows, win_size_samples_perflow, surge_thresh, stability_thresh):

    attack_periods = {}

    for prefix in attacked_prefixes:
        fid_series = [i[0] for i in rtts_dict[prefix]]
        ts_series  = [i[1] for i in rtts_dict[prefix]]
        tsr_series = [(i - math.floor(ts_series[0])) for i in ts_series]
        rtt_series = [i[2] for i in rtts_dict[prefix]]
        flows_1, time_1, rtt_1 = divide_into_windows_with_reltime(fid_series, ts_series, rtt_series, win_size_time)
        min_time, min_rtts, win_len, flows_len = extract_minimums_and_window_len(flows_1, time_1, rtt_1)

        # Calculate total active time (exclude windows with no minRTT reports)
        total_active_time = compute_active_time(min_time, min_time[0], min_time[-1], win_size_time)
        status = "normal"

        i = 0
        while i < len(min_rtts) - 2:
            t1, t2, t3 = min_time[i], min_time[i+1], min_time[i+2]
            r1, r2, r3 = min_rtts[i], min_rtts[i+1], min_rtts[i+2]
            w1, w2, w3 = win_len[i], win_len[i+1], win_len[i+2]
            f1, f2, f3 = flows_len[i], flows_len[i+1], flows_len[i+2]
            
            if t2 - t1 <= 2 * win_size_time and t3 - t2 <= 2 * win_size_time and t3 - t1 <= 3 * win_size_time \
                    and w1 >= win_size_samples_total and w2 >= win_size_samples_total and w3 >= win_size_samples_total:
                if len(f1) >= win_size_numflows and len(f2) >= win_size_numflows and len(f3) >= win_size_numflows \
                        and sum(1 for i in f1 if f1[i] >= win_size_samples_perflow) >= win_size_numflows \
                        and sum(1 for i in f2 if f2[i] >= win_size_samples_perflow) >= win_size_numflows \
                        and sum(1 for i in f3 if f3[i] >= win_size_samples_perflow) >= win_size_numflows:
                    if r2 - r1 >= surge_thresh and abs(r3 - r2) <= stability_thresh:
                        state = "attacked"
                        if prefix not in attack_periods:
                            attack_periods[prefix] = []
                        attack_time = (t3//win_size_time + 1) * win_size_time
    
                        # Find recovery time
                        for j in range(i+3, len(min_rtts)):
                            if min_rtts[j] < r2 - stability_thresh:
                                recovery_time = (min_time[j]//win_size_time + 1) * win_size_time
                                recovery_duration = recovery_time - attack_time
                                recovery_active_time = compute_active_time(min_time, attack_time, recovery_time, win_size_time)
                                attack_periods[prefix].append((
                                    attack_time, recovery_time, recovery_duration,
                                    recovery_active_time, recovery_active_time*100/total_active_time))
                                i = j  # Move index to after recovery to look for the next attack
                                state = "recovered"
                                break
    
                        if state == "attacked":
                            recovery_time = None
                            recovery_duration = None
                            recovery_active_time = compute_active_time(min_time, attack_time, min_time[-1], win_size_time)
                            attack_periods[prefix].append((
                                attack_time, recovery_time, recovery_duration,
                                recovery_active_time, recovery_active_time*100/total_active_time))
                            
            i += 1

    return attack_periods

In [ ]:
attack_periods_puservers_minsamplesperflow = compute_cost_for_attacked_prefixes_minsamplesperflow(
    rtts_prefix_puservers, attacked_prefixes_puservers_minsamplesperflow,
    window_size_time, window_size_samples_total, window_size_numflows, window_size_samples_perflow, surge_thresh, stability_thresh)

In [ ]:
cost_puservers_minsamplesperflow = [compute_total_cost(attack_periods_puservers_minsamplesperflow, prefix) for prefix in attack_periods_puservers_minsamplesperflow]

In [ ]:
cost_pct_puservers_minsamplesperflow = [compute_total_cost_pct(
    attack_periods_puservers_minsamplesperflow, prefix) for prefix in attack_periods_puservers_minsamplesperflow]

In [ ]:
total_cost_puservers_minsamplesperflow = sum(cost_puservers_minsamplesperflow)
print(f"Server in PU: Total cost = {total_cost_puservers_minsamplesperflow} ({round(total_cost_puservers_minsamplesperflow*100/total_active_time_puservers, 5)}%)")

In [ ]:
pu.cdf(cost_puservers_minsamplesperflow, xlabel="Cost (secs.)", title="Cost of false positives: Server in PU",
       plot_path="plots/cost_fp_puservers_minsamplesperflow.png")

In [ ]:
pu.cdf(cost_pct_puservers_minsamplesperflow, xlabel="Cost (%)", title="Cost of false positives: Server in PU",
       plot_path="plots/cost_pct_fp_puservers_minsamplesperflow.png")

In [ ]:
attacked_ips_path = "data/results_attacked_ips.csv"
df_attacked_ips = pd.read_csv(attacked_ips_path)

keywords = ['Verizon', 'T-Mobile', 'AT&T', 'Boost', 'U.S. Cellular']
# Create regex pattern
pattern = '|'.join(keywords)
# Filter DataFrame
df_attacked_ips_cellular = df_attacked_ips[df_attacked_ips['org'].str.contains(pattern, case=False, na=False)]
df_attacked_ips_noncellular = df_attacked_ips[~df_attacked_ips['org'].str.contains(pattern, case=False, na=False)]

print(f"All attacked prefixes: {df_attacked_ips.shape[0]}")
print(f"Cellular attacked prefixes: {df_attacked_ips_cellular.shape[0]}")
print(f"Non-cellular attacked prefixes: {df_attacked_ips_noncellular.shape[0]}")

attacked_prefixes_non_cellular = df_attacked_ips_noncellular['ip'].tolist()

In [ ]:
pu.cdfs([cost_puservers, cost_puservers_minsamplesperflow], curvelabels=["Min. 1 flow", "Min. 2 flows"],
        xlabel="Cost (secs.)", title="Cost of false +ves: Server in PU",
        plot_path="plots/cost_fp_puservers_comparative.png")

In [ ]:
pu.cdfs([cost_pct_puservers, cost_pct_puservers_minsamplesperflow], curvelabels=["Min. 1 flow", "Min. 2 flows"],
        xlabel="Cost (%)", title="Cost of false +ves: Server in PU",
        plot_path="plots/cost_pct_fp_puservers_comparative.png")

In [ ]:
cost_puservers_minsamplesperflow_noncellular = [compute_total_cost(attack_periods_puservers_minsamplesperflow, prefix)
                                        for prefix in attacked_prefixes_non_cellular]
cost_pct_puservers_minsamplesperflow_noncellular = [compute_total_cost_pct(
    attack_periods_puservers_minsamplesperflow, prefix) for prefix in attacked_prefixes_non_cellular]

In [ ]:
total_cost_puservers_minsamplesperflow_noncellular = sum(cost_puservers_minsamplesperflow_noncellular)
print(f"Server in PU: Total cost = {total_cost_puservers_minsamplesperflow_noncellular} ("
        + f"{round(total_cost_puservers_minsamplesperflow_noncellular*100/total_active_time_puservers, 5)}%)")

In [ ]:
pu.cdfs([cost_puservers, cost_puservers_minsamplesperflow, cost_puservers_minsamplesperflow_noncellular],
        curvelabels=["Min. 1 flow", "Min. 2 flows", "Min. 2 flows non-cellular"],
        xlabel="Cost (secs.)", title="Cost of false +ves: Server in PU",
        xlim=(-1,201),
        plot_path="plots/cost_fp_puservers_comparative_2.png")

In [ ]:
pu.cdfs([cost_pct_puservers, cost_pct_puservers_minsamplesperflow, cost_pct_puservers_minsamplesperflow_noncellular],
        curvelabels=["Min. 1 flow", "Min. 2 flows", "Min. 2 flows non-cellular"],
        xlabel="Cost (%)", title="Cost of false +ves: Server in PU",
        plot_path="plots/cost_pct_fp_puservers_comparative_2.png")

## Filter out unreliable prefixes

In [ ]:
from bokeh.plotting import figure, show, output_file
from bokeh.models import WMTSTileSource

# Output to an HTML file so you can view it in your browser
output_file("tile_map.html")

# URL for the CartoDB tile source
tile_url = "https://c.basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png"

# Define the map figure with an appropriate coordinate range for the whole world
p = figure(x_range=(-2000000, 6000000), y_range=(-1000000, 7000000),
           x_axis_type="mercator", y_axis_type="mercator",
           title="CartoDB Positron Tile Map")

# Add the tile source using WMTSTileSource
tile_source = WMTSTileSource(url=tile_url)
p.add_tile(tile_source)

# Show the plot
show(p)


## Find best detection parameters

### Window size (time)

#### Per flow

In [ ]:
rtt_pctiles_flow_puservers = extract_percentiles(rtts_flow_puservers)

In [ ]:
rtt_pctiles_flow_puservers = {}

for flow in rtts_flow_puservers:
    rtt_pctiles_flow_puservers[flow] = []
    rtts = 
    for p in range(101):
        rtt_pctiles_flow_puservers[flow].append(np.percentile(rtts, p))

#### Per prefix

In [ ]:
def extract_percentiles(rtts_dict):
    rtt_pctiles = {}
    for per in rtts_dict:
        rtts = [i[2] for i in rtts_dict[per]]
        if len(rtts) < 15:
            continue
        if min(rtts[:15]) > 100:
            continue
        rtt_pctiles[per] = {}
        for p in range(0, 11, 1):
            rtt_pctiles[per][p] = np.percentile(rtts, p)
    return rtt_pctiles

In [ ]:
def aggregate_percentiles(rtt_pctiles):
    agg = {}
    for per in rtt_pctiles:
        for p in range(0, 11, 1):
            if p not in agg:
                agg[p] = []
            agg[p].append(rtt_pctiles[per][p])
    return agg

In [ ]:
rtt_pctiles_prefix_puclients = extract_percentiles(rtts_prefix_puclients)

In [ ]:
rtt_pctiles_prefix_puservers = extract_percentiles(rtts_prefix_puservers)

In [ ]:
agg_rtt_pctiles_prefix_puservers = aggregate_percentiles(rtt_pctiles_prefix_puservers)

In [ ]:
data = []
labels = []
for p in agg_rtt_pctiles_prefix_puservers:
    data.append(agg_rtt_pctiles_prefix_puservers[p])
    labels.append(f"{p}")
plt.figure(figsize=(12,6))
plt.boxplot(data, labels=labels)
plt.title("Server in PU")
plt.ylabel("RTT (ms)")
plt.xlabel("Percentile")
# plt.ylim(-1, 200000)
# plt.xlim(0, 16.5)
plt.show()